# 02 - Preprocessing
Loan Default Prediction Capstone

Goal: clean, encode, and scale the data so it's ready for both the clustering step (03) and the classification step (04). Produces a single fitted scaler and a preprocessed dataframe that later notebooks will load.

## 1. Load Data

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
import os

df = pd.read_csv('../Loan_default.csv')
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


## 2. Drop Non-Predictive Columns
`LoanID` is just an identifier — no predictive value, drop it.

In [3]:
df = df.drop(columns=['LoanID'])
df.shape

(255347, 17)

## 3. Handle Missing Values (if any)
Based on EDA findings — adjust this cell if 01_eda.ipynb found nulls worth imputing rather than dropping.

In [4]:
print("Nulls before handling:")
print(df.isnull().sum().sum())

# if EDA showed minimal/no nulls, this is just a safety net
df = df.dropna()
print("Shape after dropping nulls:", df.shape)

Nulls before handling:
0
Shape after dropping nulls: (255347, 17)


## 4. Select Categorical Columns for the Streamlit Input Form
Dataset has 7 categorical columns, but only 1-2 will be exposed as live user input in the app (keeps the demo form simple). The rest are still used for training but won't be asked of the live user (defaulted to the most common category at prediction time).

Chosen for live input: **EmploymentType** and **HasCoSigner** — selected based on EDA findings showing these two had the largest spread in default rate across categories (~4.1 points and ~2.5 points respectively), making them the most behaviorally meaningful and demo-worthy inputs. MaritalStatus was considered but showed one of the weakest spreads (~2.1 points) and a less direct real-world tie to lending risk.

In [5]:
live_input_categorical = ['EmploymentType', 'HasCoSigner']
background_categorical = ['Education', 'LoanPurpose', 'HasMortgage', 'HasDependents', 'MaritalStatus']

all_categorical = live_input_categorical + background_categorical
numeric_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
                'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

target_col = 'Default'

## 5. One-Hot Encode Categorical Features

In [6]:
df_encoded = pd.get_dummies(df, columns=all_categorical, drop_first=True)
df_encoded.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Default,...,Education_Master's,Education_PhD,LoanPurpose_Business,LoanPurpose_Education,LoanPurpose_Home,LoanPurpose_Other,HasMortgage_Yes,HasDependents_Yes,MaritalStatus_Married,MaritalStatus_Single
0,56,85994,50587,520,80,4,15.23,36,0.44,0,...,False,False,False,False,False,True,True,True,False,False
1,69,50432,124440,458,15,1,4.81,60,0.68,0,...,True,False,False,False,False,True,False,False,True,False
2,46,84208,129188,451,26,3,21.17,24,0.31,1,...,True,False,False,False,False,False,True,True,False,False
3,32,31713,44799,743,0,3,7.07,24,0.23,0,...,False,False,True,False,False,False,False,False,True,False
4,60,20437,9139,633,8,4,6.51,48,0.73,0,...,False,False,False,False,False,False,False,True,False,False


## 6. Scale Numeric Features
One fitted `StandardScaler`, saved and reused across K-Means (notebook 03) and Logistic Regression (notebook 04) — keeps everything consistent, including at Streamlit prediction time.

In [7]:
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])
df_encoded[numeric_cols].describe()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio
count,2.553470e+05,2.553470e+05,2.553470e+05,2.553470e+05,2.553470e+05,2.553470e+05,2.553470e+05,2.553470e+05,2.553470e+05
mean,-1.812065e-16,3.784411e-17,4.147548e-17,2.335205e-16,9.366418e-17,-2.983007e-17,1.408024e-16,1.861040e-16,5.253654e-17
std,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00
min,-1.700995e+00,-1.732398e+00,-1.730348e+00,-1.725980e+00,-1.718715e+00,-1.343791e+00,-1.731770e+00,-1.415845e+00,-1.733149e+00
25%,-8.337635e-01,-8.642522e-01,-8.670578e-01,-8.638217e-01,-8.527470e-01,-4.485487e-01,-8.623271e-01,-7.086855e-01,-8.670336e-01
50%,-3.324207e-02,-8.547763e-04,-3.227743e-04,-1.663564e-03,1.322113e-02,-4.485487e-01,-4.938420e-03,-1.525943e-03,-9.183609e-04
75%,8.339895e-01,8.654300e-01,8.668216e-01,8.667877e-01,8.791892e-01,4.466940e-01,8.675186e-01,7.056336e-01,8.651968e-01
max,1.701221e+00,1.732408e+00,1.728108e+00,1.728946e+00,1.716292e+00,1.341937e+00,1.733948e+00,1.412793e+00,1.731312e+00


## 7. Save Preprocessed Data and Scaler
These outputs get loaded directly by 03_clustering.ipynb and 04_classification.ipynb — no need to re-run preprocessing in every notebook.

In [8]:
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

df_encoded.to_csv('../data/preprocessed_data.csv', index=False)
joblib.dump(scaler, '../models/scaler.pkl')

print("Saved: data/preprocessed_data.csv")
print("Saved: models/scaler.pkl")
print("Final shape:", df_encoded.shape)

Saved: data/preprocessed_data.csv
Saved: models/scaler.pkl
Final shape: (255347, 25)


## 8. Notes
(No model is trained in this notebook — this step only produces clean, encoded, scaled data and a saved scaler. Model training starts in 03_clustering.ipynb.)